# RealSaS Fit8 — G1→G2 Dense Field Diagnostic

Read-only diagnosis of the preserved **step-1600** scientific checkpoint. Drive auth occurs once in the first code cell. This notebook never calls `trainer.one_step()`, never takes an optimizer step, and never writes a checkpoint.

In [ ]:
from pathlib import Path
import os,sys,json,hashlib,shutil,subprocess,math,time
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
DRIVE=build('drive','v3',cache_discovery=False)

REPO_URL='https://github.com/merynz/RealSaS-OPT.git'
REPO_COMMIT='fafe328499e3c55a37492b13bff522c49861c69c'
FLEXI_URL='https://github.com/nv-tlabs/FlexiCubes.git'
FLEXI_COMMIT='4cc7d6c3d0cee83c011ce36721b81adff0dd7db6'
MANIFEST_SHA='af6a53ad13059cb1f6b69478046b38fba0cb01cdfa6e4a107a6b4c9e85dfb53f'
RECEIPT_SHA='5aadb980c0cb4cef1c584ec69367b3661d676b8edcc945e33c5f9c01d9e346ab'
FILES={'COHORT_MANIFEST.json':'1Ib5Sp4p5Nw4n-iNFvtbQmR2yKJaCc2X0','PREPARATION_RECEIPT.json':'1_Dmgt2v-UrgaHSH8RB6KhQIeKBJwGvvw','snake.vnext.pt':'1NLD77EhsUvYEDw8lY9UhY_YJoaKTmv5S','bat.vnext.pt':'1iSLfGJraOvTrP0xH1ewfU1dAKLItyRcl','trex.vnext.pt':'1YvEf5QKeI2tMwUVyqh8OTp2snVKbsv2b','horse.vnext.pt':'1A8ZpY0WStPotuuUZ_lsy45Xn3HaemTbf','spider.vnext.pt':'1WcLTcwzQk9UHILXBRLxAGf6gmN72mnYp','greenblob.vnext.pt':'1rMgxij385rRzAfzDmnaEm5Z55I6f22k2','shark.vnext.pt':'1Zlm9ubcmM_aA_7VSsQgjDRHu7P1HuNU_','george.vnext.pt':'1rxn4ZV1_9bghAHSWF_EEkHUQK-DhYoW_'}
POINTER_ID='1_Q9FQAqA2dEgKmsxE7IE5tJcYRSuno2x'
PAYLOAD_ID='1qT9L5gAqRBl4q3TSgU87Gc_ki0Q4xCvM'
SUBJECTS=('Snake','Bat','T-Rex','Horse','Spider','GreenBlob','Shark','George')
WORK=Path('/content'); REPO=WORK/'RealSaS-OPT'; FLEXI=WORK/'FlexiCubes'; PREP=WORK/'realsas_fit8_vnext_sealed_prepared'; CKPT=WORK/'fit8_step1600_diag_ckpt'; REPORT=WORK/'FIT8_G1_G2_DENSE_FIELD_DIAGNOSTIC.json'
print('DRIVE_AUTH_PASS')
print('READ_ONLY__NO_OPTIMIZER_STEP__NO_CHECKPOINT_WRITE')

In [ ]:
def sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''): h.update(b)
    return h.hexdigest()

def dl(fid,dest,name=None,expected=None):
    dest=Path(dest); dest.parent.mkdir(parents=True,exist_ok=True)
    if dest.is_file() and (expected is None or sha(dest)==expected):
        print('CACHE_HIT',dest.name,dest.stat().st_size); return dest
    meta=DRIVE.files().get(fileId=fid,fields='id,name,size',supportsAllDrives=True).execute(num_retries=5)
    if name and meta['name']!=name: raise ValueError(f"NAME_DRIFT:{name}:{meta['name']}")
    tmp=dest.with_name(dest.name+'.part'); tmp.unlink(missing_ok=True)
    req=DRIVE.files().get_media(fileId=fid,supportsAllDrives=True)
    with tmp.open('wb') as fh:
        d=MediaIoBaseDownload(fh,req,chunksize=32*1024*1024); done=False
        while not done:
            st,done=d.next_chunk(num_retries=5)
            if st: print(f'DOWNLOAD {dest.name} {st.progress()*100:6.2f}%')
    if expected and sha(tmp)!=expected:
        got=sha(tmp); tmp.unlink(missing_ok=True); raise ValueError(f'SHA_DRIFT:{dest.name}:{got}:{expected}')
    os.replace(tmp,dest); print('FILE_PASS',dest.name,dest.stat().st_size); return dest

PREP.mkdir(parents=True,exist_ok=True); CKPT.mkdir(parents=True,exist_ok=True)
mp=dl(FILES['COHORT_MANIFEST.json'],PREP/'COHORT_MANIFEST.json','COHORT_MANIFEST.json',MANIFEST_SHA)
rp=dl(FILES['PREPARATION_RECEIPT.json'],PREP/'PREPARATION_RECEIPT.json','PREPARATION_RECEIPT.json',RECEIPT_SHA)
manifest=json.loads(mp.read_text())
for r in manifest['subjects']: dl(FILES[r['shard_file']],PREP/r['shard_file'],r['shard_file'],r['shard_sha256'])
pp=dl(POINTER_ID,CKPT/'LATEST.json','LATEST.json'); pointer=json.loads(pp.read_text())
assert pointer['schema']=='RealSaS.IRISVNextFit8CheckpointPointer.v2' and int(pointer['step'])==1600
payload=Path(pointer['file']).name
dl(PAYLOAD_ID,CKPT/payload,payload,pointer['sha256'])
print('STEP1600_DURABLE_CHECKPOINT_HYDRATED_PASS',pointer['sha256'])

In [ ]:
def run(cmd,cwd=None):
    print('+',' '.join(map(str,cmd)))
    p=subprocess.run(list(map(str,cmd)),cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    if p.stdout: print(p.stdout,end='')
    if p.returncode: raise RuntimeError(p.stdout[-12000:])

if REPO.exists(): shutil.rmtree(REPO)
run(['git','clone','--filter=blob:none','--no-checkout',REPO_URL,REPO]); run(['git','checkout','--detach',REPO_COMMIT],REPO)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==REPO_COMMIT
if FLEXI.exists(): shutil.rmtree(FLEXI)
run(['git','clone','--filter=blob:none',FLEXI_URL,FLEXI]); run(['git','checkout','--detach',FLEXI_COMMIT],FLEXI)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=FLEXI,text=True).strip()==FLEXI_COMMIT
run([sys.executable,'-m','pip','install','--no-input','--progress-bar','off','-r',REPO/'requirements/mainline-ci.txt'])
sys.path.insert(0,str(REPO)); os.chdir(REPO); os.environ['REALSAS_FLEXICUBES_ROOT']=str(FLEXI)
import torch
if not torch.cuda.is_available(): raise RuntimeError('CUDA_REQUIRED')
print('SOURCE_AND_FLEXICUBES_PASS',torch.cuda.get_device_name(0))

In [ ]:
from models.iris.vnext.contracts_v1 import IRISVNextConfigV1
from models.iris.vnext.cohort_v1 import VNextFit8PolicyV1
from models.iris.vnext.cohort_v2 import IRISVNextFit8TrainerV2
cfg=IRISVNextConfigV1(); policy=VNextFit8PolicyV1()
trainer=IRISVNextFit8TrainerV2(data_root=PREP,manifest=manifest,checkpoint_root=CKPT,device='cuda',model_config=cfg,policy=policy,repo_root=REPO,flexicubes_root=FLEXI)
assert trainer.step==1600 and trainer.stage_index==0 and trainer.stage_step==1600
trainer.model.eval()
print('STEP1600_MODEL_RESTORE_PASS',trainer.step,trainer.stage.stage_id,trainer.stage_step)
print('trainer.one_step() WILL NOT BE CALLED')

In [ ]:
source=trainer.cohort.tensors

def truth_census():
    out={}; t=source['geometry_targets'].float().cpu(); print('\n=== TRUTH SIGN CENSUS ===')
    for i,s in enumerate(SUBJECTS):
        q=t[i]; r={'min':float(q.min()),'max':float(q.max()),'neg_frac':float((q<0).float().mean()),'pos_frac':float((q>0).float().mean())}; out[s]=r
        print(f"{s:10s} min={r['min']:+.6f} max={r['max']:+.6f} neg={r['neg_frac']:.4f} pos={r['pos_frac']:.4f}")
    return out

@torch.no_grad()
def probe(res,bf16,extract):
    with torch.autocast(device_type='cuda',dtype=torch.bfloat16,enabled=bf16):
        planes=trainer.model.encode_scene(source['image_tokens'],source['token_xy'],source['camera_features'])
        verts,cubes=trainer.extractor.construct_iris_grid(res); verts=verts.to(trainer.device,dtype=planes.dtype); cubes=cubes.to(trainer.device)
        grid=verts.unsqueeze(0).expand(8,-1,-1); ev=trainer.model.query_evidence(planes,grid); field=ev['field']; deformed=grid+ev['extraction_delta']; raw=None
        if extract:
            chunks=[]
            for st in range(0,int(cubes.shape[0]),2048): chunks.append(trainer.model.scene_core.cube_extraction(ev['H'][:,cubes[st:st+2048]]))
            raw=torch.cat(chunks,dim=1)
    f32=field.float(); result={'resolution':res,'bf16':bf16,'subjects':{}}; print(f"\n=== DENSE res={res} mode={'BF16' if bf16 else 'FP32'} ===")
    for i,s in enumerate(SUBJECTS):
        f=f32[i]; cv=f[cubes]; crossing=int(((cv.min(1).values<0)&(cv.max(1).values>0)).sum())
        r={'min':float(f.min()),'max':float(f.max()),'abs_min':float(f.abs().min()),'neg':int((f<0).sum()),'pos':int((f>0).sum()),'cross_cubes':crossing,'zero_cross':bool(float(f.min())<0<float(f.max()))}
        if extract:
            try:
                v,fc,ld=trainer.extractor.extract(vertices_iris=deformed[i],field=field[i],cubes=cubes,raw_parameters=raw[i],resolution=res,training=True); r.update(extract='PASS',vertices=int(v.shape[0]),faces=int(fc.shape[0]))
            except Exception as e: r.update(extract='FAIL',error=f'{type(e).__name__}:{e}',vertices=0,faces=0)
        result['subjects'][s]=r; ex=f" extract={r.get('extract','-')} V/F={r.get('vertices','-')}/{r.get('faces','-')}" if extract else ''
        print(f"{s:10s} min={r['min']:+.7f} max={r['max']:+.7f} |f|min={r['abs_min']:.2e} neg={r['neg']:6d} pos={r['pos']:6d} cross_cubes={crossing:5d}{ex}")
    return result

truth=truth_census(); fp24=probe(24,False,True); bf24=probe(24,True,True); fp32=probe(32,False,False); fp40=probe(40,False,False)

In [ ]:
def exfail(x): return [s for s,r in x['subjects'].items() if r.get('extract')!='PASS']
def zfail(x): return [s for s,r in x['subjects'].items() if (not r['zero_cross']) or r['cross_cubes']<=0]
a,b,c,d=exfail(fp24),exfail(bf24),zfail(fp32),zfail(fp40)
if not a and b: cls='BF16_AUTOCAST_EXTRACTION_PARITY_FAILURE'
elif a:
    repaired=[s for s in a if s not in c or s not in d]; persistent=[s for s in a if s in c and s in d]
    if repaired and not persistent: cls='RESOLUTION_24_TOO_COARSE_FOR_ZERO_CROSSING'
    elif persistent and not repaired: cls='G1_DENSE_FIELD_ZERO_LEVEL_SUPPORT_FAILURE'
    else: cls='MIXED_G1_FIELD_AND_RESOLUTION_FAILURE'
else: cls='ZERO_CROSSING_PRESENT__INVESTIGATE_FLEXICUBES_PARAMETER_OR_DEFORMATION_PATH'
report={'schema':'RealSaS.IRISVNextFit8G1G2DenseFieldDiagnostic.v1','authority_commit':REPO_COMMIT,'checkpoint_step':1600,'checkpoint_sha256':pointer['sha256'],'optimizer_steps_taken':0,'checkpoint_writes':0,'truth':truth,'fp32_24':fp24,'bf16_24':bf24,'fp32_32':fp32,'fp32_40':fp40,'classification':cls,'fp32_24_fail':a,'bf16_24_fail':b,'fp32_32_zero_fail':c,'fp32_40_zero_fail':d}
REPORT.write_text(json.dumps(report,sort_keys=True,indent=2)+'\n')
print('\n=== VERDICT ==='); print('classification',cls); print('FP32@24 extraction fail',a); print('BF16@24 extraction fail',b); print('FP32@32 zero-cross fail',c); print('FP32@40 zero-cross fail',d); print('REPORT',REPORT); print('REPORT_SHA256',sha(REPORT)); print('READ_ONLY_DIAGNOSTIC_COMPLETE')